# Resumen de Lectura

## 1.1 La Naturaleza como Optimizadora

La naturaleza lleva millones de años resolviendo problemas sin saberlo. A través de la selección natural, solo sobreviven los individuos mejor
adaptados a su entorno, y sus características pasan a la siguiente generación. Un ejemplo del libro: el pingüino tiene una forma corporal
tan eficiente que los ingenieros la estudiaron para diseñar submarinos.

Las mutaciones son cambios accidentales en el ADN. La mayoría son dañinas, pero de vez en cuando dan ventaja, y esas ventajas se acumulan
generación tras generación hasta producir organismos casi perfectos.

Esta observación llevó a varios científicos a trasladar la lógica evolutiva a la computación. Rechenberg propuso las estrategias
evolutivas, Fogel desarrolló la programación evolutiva y Holland formalizó los algoritmos genéticos como un modelo de adaptación.

La clave es que la naturaleza encuentra buenas soluciones sin conocer la respuesta de antemano, solo probando y conservando lo que funciona. 
Los algoritmos genéticos hacen exactamente lo mismo.

## 1.2 Un Poco de Biología

Cada ser vivo tiene un genotipo (su información genética interna) que determina su fenotipo (las características que se pueden
observar, como el color de ojos o la estatura). En los algoritmos genéticos, el genotipo es la cadena de bits que usamos para representar
una solución, y el fenotipo es lo que esa cadena significa en el problema real.

Esa cadena está formada por genes, cada uno con un valor llamado alelo. El conjunto de todos los genes forma el cromosoma, y un
grupo de individuos con sus cromosomas constituye la población.

La reproducción ocurre mediante la meiosis, donde dos cromosomas se cortan en puntos llamados quiasmas e intercambian segmentos,
produciendo hijos con características mezcladas de ambos padres. Adicionalmente, la enzima ADN-polimerasa a veces comete errores al
copiar el ADN, generando mutaciones, cambios pequeños que introducen variedad en la población.

Estos dos mecanismos, cruzamiento y mutación, son la base de los operadores genéticos en los algoritmos: el cruzamiento
combina buenas soluciones y la mutación evita que todos los individuos se vuelvan iguales.

# Implementación de las Clases

En este caso, basado en la lectura, las dos clases fundamentales del cómputo evolutivo, siendo: Individuo y Poblacion.  

In [65]:
import random
from typing import List, Callable

### Clase Individuo

Un individuo es un ser vivo dentro de una población, cada uno carga su propio ADN (el cromosoma), que es simplemente una cadena de ceros y unos. Ese ADN define qué tan buena es la solución que representa.

Atributos: 
- cromosoma, el cual es su cadena de genes (0s y 1s). En nuestro caso será una List[int].
- longitud, número de genes del cromosomam. Éste será [int].
- fitness, que representa que tan apto es, calculado por la función objetivo (entre más alto, mejor solución). Siendo éste [float].

Acciones:
- Nacer con genes aleatorios (inicializar_aleatorio)
- Ser evaluado por el entorno (calcular_fitness)
- Sufrir una mutación accidental (mutar)
- Reproducirse con otro individuo (cruzar)
- Mostrar lo que significa su ADN (obtener_fenotipo)

In [66]:
class Individuo:
    def __init__(self, longitud: int):
        self.longitud: int = longitud
        self.cromosoma: List[int] = []  
        self.fitness: float = 0.0
        self.inicializar_aleatorio()

    #  Métodos

    def inicializar_aleatorio(self) -> None:
        self.cromosoma = [random.randint(0, 1) for _ in range(self.longitud)]

    def calcular_fitness(self, funcion_objetivo: Callable) -> float:
        self.fitness = funcion_objetivo(self.cromosoma)
        return self.fitness

    def mutar(self, prob_mutacion: float = 0.01) -> None:
        for i in range(self.longitud):
            if random.random() < prob_mutacion:
                self.cromosoma[i] = 1 - self.cromosoma[i] 

    def cruzar(self, otro: 'Individuo') -> tuple:
        punto_corte = random.randint(1, self.longitud - 1)

        hijo1 = Individuo(self.longitud)
        hijo2 = Individuo(self.longitud)

        hijo1.cromosoma = self.cromosoma[:punto_corte] + otro.cromosoma[punto_corte:]
        hijo2.cromosoma = otro.cromosoma[:punto_corte] + self.cromosoma[punto_corte:]

        return hijo1, hijo2

    # Interpretación del ADN en el mundo real ([1, 0, 1, 1] -> "1011" -> 11)
    def obtener_fenotipo(self) -> int:
        return int(''.join(map(str, self.cromosoma)), 2)

    def __str__(self) -> str:
        bits = ''.join(map(str, self.cromosoma))
        return (f"Individuo con cromosoma {bits} "
                f"de fenotipo {self.obtener_fenotipo()} "
                f"y fitness {self.fitness:.4f}")

    def __repr__(self) -> str:
        return self.__str__()

### Clase Poblacion

Representa el conjunto de individuos de una generación y gestiona el proceso evolutivo.

Atributos:
- individuos, que es la lista de individuos de la generación actual. Será representado con List[Individuo].
- tamano, representando el número de individuos en la población. Éste es un [int].
- longitud_cromosoma, de qué tamaño es el ADN de cada individuo. Siendo éste [int].
- generacion, contador de generaciones transcurridas. El cual es un [int].

Acciones:
- Crear la primera generación al azar (inicializar)
- Evaluar a todos sus individuos (evaluar)
- Elegir a los más aptos para reproducirse (seleccionar_padre)
- Generar la siguiente generación (evolucionar)
- Decir quién es el mejor individuo del momento (obtener_mejor)
- Calcular qué tan bien le va a la población en general
  (obtener_fitness_promedio)

In [67]:
class Poblacion:
    def __init__(self, tamano: int, longitud_cromosoma: int):
        self.tamano: int = tamano
        self.longitud_cromosoma: int = longitud_cromosoma
        self.generacion: int = 0
        self.individuos: List[Individuo] = []
        self.inicializar()

    #  Métodos

    def inicializar(self) -> None:
        self.individuos = [Individuo(self.longitud_cromosoma)
                           for _ in range(self.tamano)]
        self.generacion = 0

    def evaluar(self, funcion_objetivo: Callable) -> None:
        for individuo in self.individuos:
            individuo.calcular_fitness(funcion_objetivo)

    # Mayor fitness = mayor probabilidad de ser elegido, pero todos tienen oportunidad
    def seleccionar_padre(self) -> 'Individuo':
        fitness_total = sum(ind.fitness for ind in self.individuos)
        if fitness_total == 0:
            return random.choice(self.individuos)

        punto = random.uniform(0, fitness_total)
        acumulado = 0.0
        for individuo in self.individuos:
            acumulado += individuo.fitness
            if acumulado >= punto:
                return individuo
        return self.individuos[-1]

    def evolucionar(self, prob_cruce: float = 0.8,
                    prob_mutacion: float = 0.01) -> None:
        nueva_generacion: List[Individuo] = []

        while len(nueva_generacion) < self.tamano:
            padre1 = self.seleccionar_padre()
            padre2 = self.seleccionar_padre()

            # Cruzamiento
            if random.random() < prob_cruce:
                hijo1, hijo2 = padre1.cruzar(padre2)
            else:
                hijo1 = Individuo(self.longitud_cromosoma)
                hijo1.cromosoma = padre1.cromosoma[:]
                hijo2 = Individuo(self.longitud_cromosoma)
                hijo2.cromosoma = padre2.cromosoma[:]

            # Mutación
            hijo1.mutar(prob_mutacion)
            hijo2.mutar(prob_mutacion)

            nueva_generacion.append(hijo1)
            if len(nueva_generacion) < self.tamano:
                nueva_generacion.append(hijo2)

        self.individuos = nueva_generacion
        self.generacion += 1

    def obtener_mejor(self) -> 'Individuo':

        return max(self.individuos, key=lambda ind: ind.fitness)

    def obtener_fitness_promedio(self) -> float:
        return sum(ind.fitness for ind in self.individuos) / self.tamano

    def __str__(self) -> str:
        return (f"Poblacion de generacion {self.generacion} "
            f"con {self.tamano} individuos, "
            f"mejor fitness de {self.obtener_mejor().fitness:.2f} "
            f"y promedio de {self.obtener_fitness_promedio():.2f}")

    def __repr__(self) -> str:
        return self.__str__()

### Caso de Prueba

In [ ]:
# Función objetivo: contar la cantidad de 1s en el cromosoma
def contar_unos(cromosoma: List[int]) -> float:
    return float(sum(cromosoma))

# Configuración
random.seed(42)
TAMANO_POBLACION   = 10
LONGITUD_CROMOSOMA = 8
GENERACIONES       = 20
PROB_CRUCE         = 0.8
PROB_MUTACION      = 0.05

# Inicialización
poblacion = Poblacion(TAMANO_POBLACION, LONGITUD_CROMOSOMA)
poblacion.evaluar(contar_unos)

print("ALGORITMO GENÉTICO")
print("-" * 65)
print(f"{'Gen':<6} {'Mejor Fitness':<16} {'Prom. Fitness':<16} {'Mejor Cromosoma'}")
print("-" * 65)
for gen in range(GENERACIONES):
    mejor = poblacion.obtener_mejor()
    promedio = poblacion.obtener_fitness_promedio()
    cromosoma_str = ''.join(map(str, mejor.cromosoma))
    print(f"{gen:<6} {mejor.fitness:<16.1f} {promedio:<16.4f} {cromosoma_str}")

    if mejor.fitness == LONGITUD_CROMOSOMA:
        print("\n¡Solución óptima encontrada!")
        break

    poblacion.evolucionar(PROB_CRUCE, PROB_MUTACION)
    poblacion.evaluar(contar_unos)

    # Imprimir la última generación generada
    if gen == GENERACIONES - 1:
        mejor = poblacion.obtener_mejor()
        promedio = poblacion.obtener_fitness_promedio()
        cromosoma_str = "".join(map(str, mejor.cromosoma))
        print(f"{gen+1:<6} {mejor.fitness:<16.1f} {promedio:<16.4f} {cromosoma_str}")

print("-" * 65)
print(f"\nMejor individuo final:")
print(f"  -{poblacion.obtener_mejor()}")
print(f"  -Encontrado en: {poblacion}")

ALGORITMO GENÉTICO
-----------------------------------------------------------------
Gen    Mejor Fitness    Prom. Fitness    Mejor Cromosoma
-----------------------------------------------------------------
0      5.0              3.5000           10110011
1      6.0              3.9000           11111001
2      6.0              4.2000           00111111
3      7.0              4.2000           11011111
4      6.0              4.2000           11011110
5      6.0              3.8000           11110011
6      6.0              4.1000           01111011
7      6.0              4.5000           11011011
8      5.0              4.6000           11110010
9      6.0              4.8000           11110011
10     7.0              5.7000           11110111
11     7.0              6.2000           11011111
12     7.0              6.2000           11110111
13     7.0              5.9000           11011111
14     7.0              5.6000           11110111
15     6.0              5.5000           1